# 마지막 주 → 다음 주 예측 (last_forecast)

`train_posttrain_model.ipynb` 로 학습해 Google Drive 에 저장한 모델(`model_posttrain.pt`)을 불러와,
`ml/test.csv` 의 **마지막 날짜**를 찾고 → 그 주의 **월요일**을 `t0` 로 잡아 한 주(t0~t6, 7개 달력일)를
입력으로 만든다(없는 날/휴장/주말은 -1 로 결측 채움).

그리고 이 입력을 바탕으로 **다음 주(t7~t13)** 를 예측한다.

## 방식
- 데이터는 모든 달력일을 포함(주말=-1). 따라서 한 주 = 월~일 연속 7일.
- `t0` = 마지막 날짜가 속한 주의 월요일. 입력 = `t0~t6`, 예측 = `t7~t13`.
- 모델은 입력 `t0~t6` 로 `t1~t7`(각 다음날) 을 내는 **1-스텝** 구조이므로,
  다음 주 전체(t7~t13)는 **자기회귀 롤링**으로 구한다:
  - t7 예측: 창 `[t0..t6]` → 마지막 스텝 argmax
  - t8 예측: 창 `[t1..t7]`(t7=방금 예측값) → 마지막 스텝 argmax
  - ... t13 까지 반복. 예측값을 다시 입력으로 넣는다.
- 주말(토/일) 위치는 시장 휴장이므로 -1 로 채운다(학습 데이터와 동일한 달력 구조 유지).
- 대상 종목: **코스피 · 삼성전자 · SK하이닉스**. 각 종목별로 출력.

## Google Drive 마운트
학습 모델(`model_posttrain.pt`)이 저장된 Drive 를 마운트한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datetime import timedelta

## Config
학습 노트북과 동일한 상수. `test.csv` 는 학습과 같은 방식(git sparse-checkout)으로 가져온다.
(로컬에 `ml/test.csv` 가 있으면 그걸 사용)

In [ ]:
HERE    = os.getcwd()
MISSING = -1          # 결측 sentinel
EPS     = 1e-6
MASK_RESCALE = True   # 학습과 동일: 마스킹 시 inverted-dropout 식 보정
DROP_P  = 0.1         # 모델 구성용(eval 에선 무의미)
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 예측 대상 종목
TARGETS = ['코스피', '삼성전자', 'SK하이닉스']

# ── test.csv 경로 확보 ──
def resolve_test_csv():
    # 1) 로컬에 이미 있으면 사용
    for p in ['ml/test.csv', 'test.csv', os.path.join(HERE, 'ml', 'test.csv')]:
        if os.path.exists(p):
            return p
    # 2) 없으면 학습 노트북과 동일하게 git 에서 sparse-checkout
    REPO_URL  = 'https://github.com/syuka-fan/stock.git'
    BRANCH    = 'main'
    DATA_REPO = os.path.join(HERE, 'stock_repo')
    FILES     = ['ml/test.csv']
    if not os.path.isdir(os.path.join(DATA_REPO, '.git')):
        subprocess.run(['git', 'clone', '--filter=blob:none', '--sparse',
                        '--depth', '1', '-b', BRANCH, REPO_URL, DATA_REPO], check=True)
    subprocess.run(['git', '-C', DATA_REPO, 'sparse-checkout', 'set',
                    '--no-cone', *FILES], check=True)
    return os.path.join(DATA_REPO, 'ml', 'test.csv')

TEST_CSV = resolve_test_csv()
print('DEVICE =', DEVICE)
print('TEST_CSV =', TEST_CSV)

## 모델 정의 (학습 노트북과 동일)
체크포인트를 로드하려면 학습 때와 같은 모듈 구조가 필요하다.

In [ ]:
class RMSNorm(nn.Module):
    """표준 Root Mean Square Normalization (학습 가능 gain)."""
    def __init__(self, dim, eps=EPS):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


def masked_rms_norm(x, mask, rescale=MASK_RESCALE, eps=EPS):
    """입력 RMSNorm + 마스킹. -1(결측) 위치는 RMS 계산에서 제외 후 0."""
    valid = mask
    cnt = valid.sum(dim=-1, keepdim=True).clamp_min(1.0)
    xz = x * valid
    ms = (xz.pow(2).sum(dim=-1, keepdim=True)) / cnt
    rms = torch.sqrt(ms + eps)
    xn = (xz / rms) * valid
    if rescale:
        xn = xn * (x.shape[-1] / cnt)
    return xn


class DenseForecaster(nn.Module):
    """마스킹 RMSNorm → MLP → (OUT_LEN × vocab) 로짓."""
    def __init__(self, in_len, out_len, vocab, hidden, drop_p):
        super().__init__()
        self.out_len, self.vocab = out_len, vocab
        layers, d = [], in_len
        for h in hidden:
            layers += [nn.Linear(d, h), RMSNorm(h), nn.GELU(), nn.Dropout(drop_p)]
            d = h
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(d, out_len * vocab)

    def forward(self, x, mask):
        h = masked_rms_norm(x, mask)
        h = self.backbone(h)
        logits = self.head(h)
        return logits.view(-1, self.out_len, self.vocab)

## 학습된 모델 로드 (Google Drive)
`train_posttrain_model.ipynb` 가 저장한 `model_posttrain.pt` 를 불러온다.
(없으면 `best_model.pt` 로 대체)

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/stock_model'
CANDIDATES = [os.path.join(DRIVE_DIR, 'model_posttrain.pt'),
              os.path.join(DRIVE_DIR, 'best_model.pt')]
CKPT_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert CKPT_PATH is not None, (
    f'모델 체크포인트를 찾을 수 없습니다: {CANDIDATES}\n'
    '  → train_posttrain_model.ipynb 를 먼저 실행해 Drive 에 저장하세요.')

ckpt       = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
cfg        = ckpt['config']
vocab_vals = np.asarray(ckpt['vocab_vals']).astype(np.int64)  # 클래스 인덱스 → 원본 정수값
IN_LEN     = cfg['IN_LEN']
OUT_LEN    = cfg['OUT_LEN']

model = DenseForecaster(cfg['IN_LEN'], cfg['OUT_LEN'], cfg['vocab'],
                        cfg['HIDDEN'], DROP_P).to(DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'모델 로드 완료 ← {CKPT_PATH}')
print(f'IN_LEN={IN_LEN}  OUT_LEN={OUT_LEN}  vocab={len(vocab_vals)}')

## 마지막 날짜 → 마지막 월요일(t0) → 한 주 입력(t0~t6)
- 마지막 날짜가 속한 주의 월요일을 `t0` 로 잡는다.
- `t0~t6`(월~일 7개 달력일) 각 날짜의 종가를 조회. 데이터가 없거나 -1 이면 결측(-1).

In [ ]:
df = pd.read_csv(TEST_CSV, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

last_date = df['Date'].max().normalize()
t0 = (last_date - timedelta(days=int(last_date.weekday()))).normalize()  # 그 주의 월요일 (Mon=0)
calendar = [t0 + timedelta(days=k) for k in range(IN_LEN + OUT_LEN)]     # t0 ~ t13

print(f'마지막 날짜  = {last_date.date()} ({last_date.strftime("%A")})')
print(f't0(월요일)   = {t0.date()} ({t0.strftime("%A")})')
print(f'입력 구간    = t0~t{IN_LEN-1}: {calendar[0].date()} ~ {calendar[IN_LEN-1].date()}')
print(f'예측 구간    = t{IN_LEN}~t{IN_LEN+OUT_LEN-1}: {calendar[IN_LEN].date()} ~ {calendar[-1].date()}')

# 날짜 → 행 조회용 맵
date_to_row = {d.normalize(): i for i, d in enumerate(df['Date'])}

def build_input(stock):
    """t0~t6 의 종가 배열(round 후 int, 없거나 -1 이면 -1)."""
    vals = []
    for k in range(IN_LEN):
        d = calendar[k]
        i = date_to_row.get(d.normalize())
        if i is None:
            vals.append(MISSING); continue
        v = pd.to_numeric(df.loc[i, stock], errors='coerce')
        if pd.isna(v):
            vals.append(MISSING)
        else:
            iv = int(np.rint(v))
            vals.append(iv if iv != MISSING else MISSING)
    return np.array(vals, dtype=np.int64)

## 롤링 예측 (t7~t13)
창 `[t_{k-7}..t_{k-1}]` → 모델 → **마지막 스텝** argmax = `t_k` 예측값.
예측값을 다시 다음 창에 넣어 t13 까지 진행. 주말(토/일)은 휴장이므로 -1.

In [ ]:
@torch.no_grad()
def rolling_forecast(seed_in):
    """seed_in: (IN_LEN,) int 입력(t0~t6). 반환: (IN_LEN+OUT_LEN,) int 전체 시퀀스(t0~t13)."""
    seq = list(int(v) for v in seed_in) + [None] * OUT_LEN
    for k in range(IN_LEN, IN_LEN + OUT_LEN):
        day = calendar[k]
        if day.weekday() >= 5:          # 토(5)/일(6) → 휴장
            seq[k] = MISSING
            continue
        window = np.array(seq[k - IN_LEN:k], dtype=np.float32)   # 직전 IN_LEN 일
        mask   = (window != MISSING).astype(np.float32)
        if mask.sum() == 0:             # 창이 전부 결측이면 예측 불가
            seq[k] = MISSING
            continue
        x = torch.from_numpy(window).unsqueeze(0).to(DEVICE)
        m = torch.from_numpy(mask).unsqueeze(0).to(DEVICE)
        logits = model(x, m)            # (1, OUT_LEN, V)
        last_idx = int(logits[0, -1].argmax().item())  # 마지막 스텝 = 창 다음날 = t_k
        seq[k] = int(vocab_vals[last_idx])
    return seq

## 종목별 예측 결과 출력
코스피 · 삼성전자 · SK하이닉스 각각에 대해 입력(t0~t6)과 예측(t7~t13)을 표로 출력.

In [ ]:
def label(v):
    return '결측/휴장' if v == MISSING else f'{v:,}'

for stock in TARGETS:
    seed = build_input(stock)
    seq  = rolling_forecast(seed)

    rows = []
    for k in range(IN_LEN + OUT_LEN):
        d = calendar[k]
        rows.append({
            'step': f't{k}',
            'Date': d.date().isoformat(),
            'Day' : d.strftime('%a'),
            '구분' : '입력' if k < IN_LEN else '예측',
            '종가' : label(seq[k]),
        })
    out = pd.DataFrame(rows)

    print('=' * 56)
    print(f'[{stock}]  t0={t0.date()} (월)  →  다음주 예측')
    print('=' * 56)
    print(out.to_string(index=False))

    fc = [seq[k] for k in range(IN_LEN, IN_LEN + OUT_LEN)]
    fc_days = [f"{calendar[k].strftime('%m/%d')}({calendar[k].strftime('%a')})={label(seq[k])}"
               for k in range(IN_LEN, IN_LEN + OUT_LEN)]
    print('\n다음주(t7~t13) 예측:', ' | '.join(fc_days))
    print()